# 02c — Spatial Calibration
Map each camera's pixel coordinates to a shared physical world space (centimetres).

## How it works
1. You enter the **physical dimensions** of each chamber and the **overlap** between adjacent cameras.
2. The notebook extracts one reference frame from each camera and displays it.
3. You **click (or type) the pixel coordinates** of the four visible chamber corners in each frame.
4. OpenCV's `findHomography` computes the 3 × 3 mapping matrix for each camera.
5. Matrices are saved to Drive as `H_cam{N}.npy` and the world layout as `camera_layout.json`.

**Run this notebook once per experimental setup** (i.e., once after the cameras are positioned and before any recording sessions).

---
## Step 1 — Fill in physical dimensions below, then Run All

In [ ]:
# ===== CONFIGURATION — edit this cell =====

GITHUB_REPO_URL = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH      = "main"

DRIVE_ROOT    = "/content/drive/Shareddrives/3R.Data/1P.SPRIND.Data/POC/Behavior"
SESSION_FOLDER = f"{DRIVE_ROOT}/260608.00000009"   # any session with all 4 cameras present
CALIB_DIR     = f"{DRIVE_ROOT}/calibration"        # where H_cam*.npy files are saved

# ── Physical dimensions ──────────────────────────────────────────────────────
# For each camera, enter the size of the chamber AREA VISIBLE IN THE VIDEO.
# 'width_cm'  = dimension along the direction of travel  (Camera 1 → Camera 4)
# 'height_cm' = dimension perpendicular to travel
# Measure with a tape; confirm the corners you will click correspond to the
# full width and height of what the camera can see.

CHAMBER_DIMS = {
    1: {"width_cm": 200.0, "height_cm": 150.0},   # <-- SET THESE
    2: {"width_cm": 200.0, "height_cm": 150.0},
    3: {"width_cm": 200.0, "height_cm": 150.0},
    4: {"width_cm": 200.0, "height_cm": 150.0},
}

# Physical overlap between adjacent cameras along direction of travel (cm).
# If Camera 1 and Camera 2 share 20 cm of visible area, enter 20.0.
OVERLAPS_CM = {
    (1, 2): 20.0,   # <-- SET THESE
    (2, 3): 20.0,
    (3, 4): 20.0,
}

# Corner click ORDER (must match the world corners computed below):
#   1 = Top-Left     2 = Top-Right
#   3 = Bottom-Right 4 = Bottom-Left
# "Top" = low Y value in world coords (far from the animal's home cage end)
CORNER_ORDER = ["top_left", "top_right", "bottom_right", "bottom_left"]


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os, sys

REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print("Repo ready.")


In [ ]:
!pip install --quiet opencv-python-headless numpy matplotlib ipympl
!apt-get install -y -qq ffmpeg > /dev/null 2>&1
print("Dependencies installed.")


---
## Step 2 — Compute world coordinate layout

The notebook builds the global coordinate grid from your physical dimensions.
Check the printed layout and verify the numbers match your actual setup.

In [ ]:
from src.calibration.homography import build_world_layout, save_camera_layout

layout = build_world_layout(CHAMBER_DIMS, OVERLAPS_CM)
print("Global world coordinate layout")
print("-" * 50)
for cam_info in layout:
    cam = cam_info['camera']
    print(f"Camera {cam}: "
          f"X [{cam_info['x_min']:.1f}, {cam_info['x_max']:.1f}] cm  |  "
          f"Y [{cam_info['y_min']:.1f}, {cam_info['y_max']:.1f}] cm")
    for i, (cx, cy) in enumerate(cam_info['world_corners']):
        print(f"  Corner {i+1} ({CORNER_ORDER[i]}): ({cx:.1f}, {cy:.1f}) cm")
    print()

total_x = max(e['x_max'] for e in layout)
print(f"Total arena length: {total_x:.1f} cm")
print(f"Arena height      : {CHAMBER_DIMS[1]['height_cm']:.1f} cm")


---
## Step 3 — Extract reference frames

One frame (after the animal has settled) is extracted from each camera.
These will be displayed for you to click on.

In [ ]:
import subprocess, json, cv2, numpy as np
from pathlib import Path

def get_one_frame(video_path: Path, frame_idx: int = 150) -> np.ndarray | None:
    """Decode one frame from a video using ffmpeg."""
    cmd = [
        "ffmpeg", "-y", "-i", str(video_path),
        "-vf", f"select=eq(n\\,{frame_idx})",
        "-vframes", "1",
        "-f", "image2pipe", "-pix_fmt", "rgb24", "-vcodec", "rawvideo", "pipe:1"
    ]
    r = subprocess.run(cmd, capture_output=True, timeout=60)
    if r.returncode != 0 or not r.stdout:
        return None
    # Get frame size from ffprobe
    probe = subprocess.run(
        ["ffprobe", "-v", "quiet", "-print_format", "json",
         "-show_streams", str(video_path)],
        capture_output=True, text=True, timeout=30
    )
    info = json.loads(probe.stdout)
    for s in info.get("streams", []):
        if s.get("codec_type") == "video":
            w, h = int(s["width"]), int(s["height"])
            frame = np.frombuffer(r.stdout, dtype=np.uint8).reshape(h, w, 3)
            return frame
    return None

# Find one video per camera in the session folder
import re
video_exts = {".asf", ".mp4", ".avi", ".mov", ".mkv"}
session_path = Path(SESSION_FOLDER)

cam_frames: dict[int, np.ndarray] = {}
cam_video:  dict[int, str]         = {}

for vf in sorted(session_path.rglob("*")):
    if vf.suffix.lower() not in video_exts:
        continue
    stem  = vf.stem
    parts = re.split(r"[-_]+", stem)
    cam   = 0
    for p in reversed(parts):
        try:
            n = int(p)
            if 1 <= n <= 4:
                cam = n; break
        except ValueError:
            pass
    if cam == 0 or cam in cam_frames:
        continue
    print(f"  Camera {cam}: extracting frame from {vf.name} …", end=" ")
    frame = get_one_frame(vf, frame_idx=150)
    if frame is not None:
        cam_frames[cam] = frame
        cam_video[cam]  = vf.name
        print("OK", frame.shape)
    else:
        print("FAILED — trying first frame …")
        frame = get_one_frame(vf, frame_idx=0)
        if frame is not None:
            cam_frames[cam] = frame
            cam_video[cam]  = vf.name
            print("  OK", frame.shape)

print(f"\nFrames ready for cameras: {sorted(cam_frames)}")


---
## Step 4 — Click chamber corners in each frame

Run the cell below to display each camera's frame interactively.
**Click the 4 corners** in the order shown on the title bar:

    1 → Top-Left      2 → Top-Right
    3 → Bottom-Right  4 → Bottom-Left

A red ✕ marker and coordinate label appear after each click.
If you make a mistake, re-run the cell for that camera.

> **Tip:** Zoom in with the magnifier tool before clicking for sub-pixel accuracy.

In [ ]:
%matplotlib widget
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

PIXEL_POINTS = {}   # filled by interactive clicking below

def pick_corners_for_camera(cam: int) -> list[tuple[int, int]]:
    frame = cam_frames.get(cam)
    if frame is None:
        print(f"No frame for camera {cam}")
        return []

    fig, ax = plt.subplots(figsize=(14, 8))
    ax.imshow(frame)
    ax.set_title(
        f"Camera {cam} — {cam_video[cam]}\n"
        f"Click corners in order: 1=TL  2=TR  3=BR  4=BL",
        fontsize=11
    )
    ax.set_xlabel("Pixel X →", fontsize=9)
    ax.set_ylabel("Pixel Y ↓", fontsize=9)

    pts = []
    labels = ["1·TL", "2·TR", "3·BR", "4·BL"]
    colours = ["#E05252", "#E09952", "#52A8E0", "#52E07A"]

    def onclick(event):
        if event.inaxes != ax or len(pts) >= 4:
            return
        x, y = int(round(event.xdata)), int(round(event.ydata))
        pts.append((x, y))
        ax.plot(x, y, "x", color=colours[len(pts)-1],
                markersize=14, markeredgewidth=2.5)
        ax.text(x + 8, y - 8, f"{labels[len(pts)-1]} ({x},{y})",
                color=colours[len(pts)-1], fontsize=9, fontweight="bold")
        fig.canvas.draw()
        print(f"  Corner {len(pts)} ({labels[len(pts)-1]}): pixel ({x}, {y})")
        if len(pts) == 4:
            print(f"Camera {cam}: all 4 corners recorded ✓")

    fig.canvas.mpl_connect("button_press_event", onclick)
    plt.tight_layout()
    plt.show()
    return pts

# Pick corners for each camera sequentially
for cam in sorted(cam_frames):
    print(f"\n=== Camera {cam} ===")
    PIXEL_POINTS[cam] = pick_corners_for_camera(cam)


**If interactive clicking doesn't work** in your Colab session,
run the fallback cell below instead: display the image, note the pixel
coordinates by hovering, and type them in manually.

In [ ]:
# ── FALLBACK: manual coordinate entry ─────────────────────────────────────
# Uncomment and fill in if the interactive cell above didn't record coordinates.

# import matplotlib.pyplot as plt
# for cam in sorted(cam_frames):
#     plt.figure(figsize=(14, 8))
#     plt.imshow(cam_frames[cam])
#     plt.title(f"Camera {cam} — hover to see pixel (x, y) in bottom-right status bar")
#     plt.tight_layout()
#     plt.savefig(f"/tmp/cam{cam}_frame.png", dpi=120)
#     plt.show()

# PIXEL_POINTS = {
#     1: [(x1, y1), (x2, y2), (x3, y3), (x4, y4)],   # TL TR BR BL for Cam1
#     2: [(x1, y1), (x2, y2), (x3, y3), (x4, y4)],
#     3: [(x1, y1), (x2, y2), (x3, y3), (x4, y4)],
#     4: [(x1, y1), (x2, y2), (x3, y3), (x4, y4)],
# }

print("Current PIXEL_POINTS:", PIXEL_POINTS)


---
## Step 5 — Compute and save homographies

In [ ]:
from pathlib import Path
import numpy as np
from src.calibration.homography import (
    compute_homography, save_homography,
    save_camera_layout, reprojection_error
)

calib_path = Path(CALIB_DIR)
calib_path.mkdir(parents=True, exist_ok=True)

# Build a lookup: camera → world corners
world_corners_by_cam = {e["camera"]: e["world_corners"] for e in layout}

homographies = {}
errors       = {}

for cam in sorted(PIXEL_POINTS):
    pix_pts   = PIXEL_POINTS[cam]
    world_pts = world_corners_by_cam.get(cam)

    if not pix_pts or len(pix_pts) < 4:
        print(f"Camera {cam}: fewer than 4 points — skipped")
        continue
    if world_pts is None:
        print(f"Camera {cam}: no world layout entry — skipped")
        continue

    print(f"\n=== Camera {cam} ===")
    H = compute_homography(
        np.float32(pix_pts),
        np.float32(world_pts),
    )
    homographies[cam] = H

    err = reprojection_error(np.float32(pix_pts), np.float32(world_pts), H)
    errors[cam] = err
    print(f"  Mean reprojection error: {err:.2f} cm  {'✓ good' if err < 5 else '⚠ check corners'}")

    path = save_homography(H, cam, calib_path)
    print(f"  Saved: {path}")

# Save layout
layout_path = save_camera_layout(layout, calib_path)
print(f"\nCamera layout saved: {layout_path}")


---
## Step 6 — Visual verification

Warp each reference frame into world coordinates and overlay it.
A correctly calibrated image should look like a flat, undistorted top-down view.

In [ ]:
import matplotlib.pyplot as plt
import cv2, numpy as np

# World canvas resolution: 1 cm = 4 pixels
SCALE = 4
total_w_cm = max(e["x_max"] for e in layout)
total_h_cm = max(e["y_max"] for e in layout)
canvas_w   = int(total_w_cm * SCALE) + 1
canvas_h   = int(total_h_cm * SCALE) + 1

canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
canvas[:] = 30   # dark background

for cam, H in homographies.items():
    frame = cam_frames[cam]
    h_px, w_px = frame.shape[:2]

    # Build mapping from canvas pixel → camera pixel (inverse homography)
    H_inv = np.linalg.inv(H)

    # Scale factor: canvas pixel → world cm → camera pixel
    # Canvas pixel (cx, cy) → world (cx/SCALE, cy/SCALE) → camera px via H_inv
    H_canvas = H_inv.copy()
    # Adjust for SCALE: pre-multiply x and y by 1/SCALE
    S = np.diag([1/SCALE, 1/SCALE, 1.0])
    H_warp = H_inv @ S

    warped = cv2.warpPerspective(frame, H_warp, (canvas_w, canvas_h))
    # Blend only non-zero pixels
    mask = warped.sum(axis=2) > 0
    canvas[mask] = warped[mask]

# Draw camera boundary lines and labels
for cam_info in layout:
    cam = cam_info["camera"]
    corners_w = [(int(x * SCALE), int(y * SCALE))
                 for x, y in cam_info["world_corners"]]
    for i in range(4):
        cv2.line(canvas, corners_w[i], corners_w[(i+1) % 4], (0, 220, 100), 1)
    cx = int((cam_info["x_min"] + cam_info["x_max"]) / 2 * SCALE)
    cy = int(cam_info["y_max"] / 2 * SCALE)
    cv2.putText(canvas, f"Cam{cam}", (cx - 20, cy),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 220, 100), 2)

fig, ax = plt.subplots(figsize=(16, 6))
ax.imshow(canvas[:, :, ::-1])   # BGR→RGB
ax.set_title("World-space composite — check that corners align correctly", fontsize=12)
ax.set_xlabel(f"World X (px at {SCALE} px/cm) →")
ax.set_ylabel(f"World Y (px at {SCALE} px/cm) ↓")

# Ruler ticks every 50 cm
import matplotlib.ticker as ticker
ax.xaxis.set_major_locator(ticker.MultipleLocator(50 * SCALE))
ax.yaxis.set_major_locator(ticker.MultipleLocator(50 * SCALE))
ax.set_xticklabels([f"{int(t / SCALE)}cm" for t in ax.get_xticks()])
ax.set_yticklabels([f"{int(t / SCALE)}cm" for t in ax.get_yticks()])
plt.tight_layout()

out_img = calib_path / "world_composite.png"
plt.savefig(str(out_img), dpi=150)
plt.show()
print(f"Saved: {out_img}")


In [ ]:
print("=" * 55)
print("Calibration summary")
print("=" * 55)
for cam in sorted(homographies):
    print(f"  Camera {cam}  |  reprojection error: {errors.get(cam, '?'):.2f} cm")

files = list(Path(CALIB_DIR).glob("*"))
print(f"\nFiles saved to {CALIB_DIR}:")
for f in sorted(files):
    print(f"  {f.name}")

print("\nNext step → run 04_Pose_Inference.ipynb (if not done yet),")
print("then 04c_Trajectory_Stitch.ipynb to assemble the global trajectory.")
